# Analytical Calendar and Sales Preparation

## Objective

Build the daily analytical calendar and controlled monthly sales datasets
from standardized source files.

## Governance rules

- Preserve missing sales dates; do not impute sales.
- Preserve missing transaction records; do not assume zero transactions.
- Aggregate holiday events to date level before joining to the calendar.
- Reconcile monthly sales against the source sales total.
- Retain partial August 2017 in the full monthly dataset, but exclude it
  from the complete-month forecasting dataset.

This notebook reuses the existing preparation logic from the original
profiling notebook. The original notebook and backup remain unchanged.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Locate the project root independently of the notebook's working directory.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root.")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load standardized datasets and parse dates explicitly.
train = pd.read_csv(
    PROCESSED_DIR / "train_standardized.csv",
    parse_dates=["date"],
)
transactions = pd.read_csv(
    PROCESSED_DIR / "transactions_standardized.csv",
    parse_dates=["date"],
)
holidays = pd.read_csv(
    PROCESSED_DIR / "holidays_events_standardized.csv",
    parse_dates=["date"],
)

sales_start = train["date"].min()
sales_end = train["date"].max()

# Preserve the source event records within the sales date range.
holiday_working = holidays[
    holidays["date"].between(sales_start, sales_end)
].copy()

# Create the one-row-per-date calendar backbone.
calendar = pd.DataFrame({
    "date": pd.date_range(sales_start, sales_end, freq="D")
})

print("Project root:", PROJECT_ROOT)
print("Sales rows:", f"{len(train):,}")
print("Transaction rows:", f"{len(transactions):,}")
print("Holiday/event rows:", f"{len(holidays):,}")
print("Holiday/event rows in sales period:", f"{len(holiday_working):,}")
print("Sales date range:", sales_start.date(), "to", sales_end.date())
print("Initial calendar rows:", f"{len(calendar):,}")


### Step 3.5.2.2 — Add Basic Date Attributes

In [ ]:
# ============================================================
# Step 3.5.2.2 — Add Basic Date Attributes
# ============================================================

calendar["year"] = calendar["date"].dt.year

calendar["quarter"] = calendar["date"].dt.quarter

calendar["month"] = calendar["date"].dt.month

calendar["month_name"] = calendar["date"].dt.month_name()

calendar["week"] = calendar["date"].dt.isocalendar().week.astype("int64")

calendar["day"] = calendar["date"].dt.day

calendar["day_of_week"] = calendar["date"].dt.dayofweek

calendar["day_name"] = calendar["date"].dt.day_name()

calendar["is_weekend"] = calendar["day_of_week"].isin([5, 6])

calendar["days_from_start"] = (
    calendar["date"] - calendar["date"].min()
).dt.days

print("Date attributes added successfully.")
print("Rows:", len(calendar))
print("Columns:", len(calendar.columns))

calendar.head()

### Step 3.5.2.3 — Validate the calendar structure

In [ ]:
# ============================================================
# Step 3.5.2.3 — Calendar Structural Validation
# ============================================================

print("Calendar structural validation")
print("-" * 40)

print("Expected rows:", 1688)
print("Actual rows:", len(calendar))

print("\nUnique dates:", calendar["date"].nunique())

print(
    "Duplicate dates:",
    calendar["date"].duplicated().sum()
)

print(
    "Missing dates:",
    1688 - calendar["date"].nunique()
)

print(
    "Date sorted:",
    calendar["date"].is_monotonic_increasing
)

### Step 3.5.2.4 — Validate the time index

In [ ]:
print("Time-index validation")
print("-" * 40)

print("Minimum days_from_start:", calendar["days_from_start"].min())
print("Maximum days_from_start:", calendar["days_from_start"].max())

print(
    "Unique time-index values:",
    calendar["days_from_start"].nunique()
)

print(
    "Sequential time index:",
    calendar["days_from_start"].is_monotonic_increasing
)

### Phase 3.5.3 — Holiday/Event Aggregation
### Step 3.5.3.1 — Create a working holiday dataset

In [ ]:
# ============================================================
# Step 3.5.3 — Holiday/Event Aggregation
# ============================================================

holiday_working = holidays.copy()

# Keep only holiday/event records within the analytical period
holiday_working = holiday_working[
    holiday_working["date"].between(
        sales_start,
        sales_end
    )
].copy()

print("Holiday/event working dataset created.")
print("Rows:", len(holiday_working))
print("Unique dates:", holiday_working["date"].nunique())
print("Minimum date:", holiday_working["date"].min())
print("Maximum date:", holiday_working["date"].max())

### Step 3.5.3.2 — Inspect Event Types and Locales

In [ ]:
# ============================================================
# Step 3.5.3.2 — Inspect Holiday/Event Categories
# ============================================================

print("Holiday/Event Types")
print("-" * 40)

print(holiday_working["type"].value_counts().sort_index())

print("\nHoliday/Event Locales")
print("-" * 40)

print(holiday_working["locale"].value_counts().sort_index())

### Step 3.5.3.3 — Check category combinations

In [ ]:
print("Type × Locale")
print("-" * 40)

type_locale = pd.crosstab(
    holiday_working["type"],
    holiday_working["locale"]
)

print(type_locale)

### Step 3.5.3.4 — Check multiple records per date

In [ ]:
date_event_counts = (
    holiday_working
    .groupby("date")
    .size()
    .sort_values(ascending=False)
)

print("Holiday/Event records per date")
print("-" * 40)

print(date_event_counts.value_counts().sort_index())

print("\nDates with multiple holiday/event records:",
      (date_event_counts > 1).sum())

print("\nMaximum records on one date:",
      date_event_counts.max())

### Step 3.5.3.5 — Create the One-Row-Per-Date Holiday Summary

In [ ]:
# ============================================================
# Step 3.5.3.5 — Create Date-Level Holiday Summary
# ============================================================

holiday_summary = (
    holiday_working
    .groupby("date")
    .agg(
        holiday_event_count=("date", "size"),

        has_holiday=("type", lambda x: (x == "Holiday").any()),
        has_event=("type", lambda x: (x == "Event").any()),
        has_additional=("type", lambda x: (x == "Additional").any()),
        has_bridge=("type", lambda x: (x == "Bridge").any()),
        has_transfer=("type", lambda x: (x == "Transfer").any()),
        has_work_day=("type", lambda x: (x == "Work Day").any()),

        has_national_holiday=("locale", lambda x: (x == "National").any()),
        has_regional_holiday=("locale", lambda x: (x == "Regional").any()),
        has_local_holiday=("locale", lambda x: (x == "Local").any())
    )
    .reset_index()
)

print("Holiday summary created.")
print("Rows:", len(holiday_summary))
print("Columns:", len(holiday_summary.columns))

holiday_summary.head()

### Step 3.5.3.6 — Validate the aggregation

In [ ]:
# ============================================================
# Step 3.5.3.6 — Validate Holiday Summary
# ============================================================

print("Holiday summary validation")
print("-" * 40)

print("Expected unique dates:", holiday_working["date"].nunique())
print("Actual rows:", len(holiday_summary))

print(
    "Unique dates:",
    holiday_summary["date"].nunique()
)

print(
    "Duplicate dates:",
    holiday_summary["date"].duplicated().sum()
)

print(
    "Minimum date:",
    holiday_summary["date"].min()
)

print(
    "Maximum date:",
    holiday_summary["date"].max()
)

### Step 3.5.3.7 — Prove No Holiday/Event Records Were Lost

In [ ]:
# ============================================================
# Step 3.5.3.7 — Validate Event Record Preservation
# ============================================================

source_record_count = len(holiday_working)

aggregated_record_count = (
    holiday_summary["holiday_event_count"].sum()
)

print("Holiday/Event record preservation")
print("-" * 40)

print("Source records:", source_record_count)
print("Aggregated record count:", aggregated_record_count)

print(
    "Records preserved:",
    source_record_count == aggregated_record_count
)

### Step 3.5.3.8 — Check Boolean Indicators

In [ ]:
indicator_columns = [
    "has_holiday",
    "has_event",
    "has_additional",
    "has_bridge",
    "has_transfer",
    "has_work_day",
    "has_national_holiday",
    "has_regional_holiday",
    "has_local_holiday"
]

print("Boolean indicator validation")
print("-" * 40)

for column in indicator_columns:
    print(
        f"{column}:",
        sorted(holiday_summary[column].unique().tolist())
    )

### Step 3.5.3.9 — Check the highest-multiplicity dates

In [ ]:
print("Dates with 3 or more holiday/event records")
print("-" * 40)

high_multiplicity = holiday_summary[
    holiday_summary["holiday_event_count"] >= 3
].sort_values(
    ["holiday_event_count", "date"],
    ascending=[False, True]
)

print(high_multiplicity)

### Step 3.5.4 — Merge Holiday Features into the Analytical Calendar

In [ ]:
# ============================================================
# Step 3.5.4.1 — Merge Holiday Summary into Calendar
# ============================================================

calendar = calendar.merge(
    holiday_summary,
    on="date",
    how="left",
    validate="one_to_one"
)

print("Holiday summary merged successfully.")
print("Rows:", len(calendar))
print("Columns:", len(calendar.columns))

### Step 3.5.4.2 — Fill dates with no holiday/event

In [ ]:
# ============================================================
# Step 3.5.4.2 — Fill Non-Event Dates
# ============================================================

indicator_columns = [
    "has_holiday",
    "has_event",
    "has_additional",
    "has_bridge",
    "has_transfer",
    "has_work_day",
    "has_national_holiday",
    "has_regional_holiday",
    "has_local_holiday"
]

count_columns = [
    "holiday_event_count"
]

calendar[indicator_columns] = (
    calendar[indicator_columns]
    .fillna(False)
)

calendar[count_columns] = (
    calendar[count_columns]
    .fillna(0)
    .astype("int64")
)

print("Non-event dates handled successfully.")

### Step 3.5.4.3 — Create the combined event indicator

In [ ]:
calendar["has_holiday_event"] = (
    calendar["holiday_event_count"] > 0
)

print("Combined holiday/event indicator created.")

print(
    calendar["has_holiday_event"].value_counts()
)

In [ ]:
# ============================================================
# Step 3.5.5.1 — Structural QA
# ============================================================

print("Expected rows:", 1688)
print("Actual rows:", len(calendar))

print("Unique dates:", calendar["date"].nunique())
print("Duplicate dates:", calendar["date"].duplicated().sum())

print("\nDate range:")
print("Min:", calendar["date"].min())
print("Max:", calendar["date"].max())

In [ ]:
# ============================================================
# Step 3.5.5.2 — Holiday Event Count QA
# ============================================================

print("Total aggregated event records:",
      calendar["holiday_event_count"].sum())

print("Expected source records:",
      len(holiday_working))

print(
    "\nEvent count distribution:"
)

print(
    calendar.loc[
        calendar["holiday_event_count"] > 0,
        "holiday_event_count"
    ].value_counts().sort_index()
)

In [ ]:
# ============================================================
# Step 3.5.5.3 — Christmas Date QA
# ============================================================

christmas_dates = pd.to_datetime([
    "2013-12-25",
    "2014-12-25",
    "2015-12-25",
    "2016-12-25"
])

christmas_check = calendar[
    calendar["date"].isin(christmas_dates)
][[
    "date",
    "has_holiday_event",
    "holiday_event_count",
    "has_holiday",
    "has_national_holiday"
]]

print(christmas_check)

In [ ]:
# ============================================================
# Step 3.5.5.4 — Final Holiday Merge QA
# ============================================================

holiday_merge_checks = {
    "rows_unchanged": len(calendar) == 1688,
    "dates_unique": calendar["date"].nunique() == 1688,
    "no_duplicate_dates": calendar["date"].duplicated().sum() == 0,
    "event_records_preserved": (
        calendar["holiday_event_count"].sum()
        == len(holiday_working)
    ),
    "all_christmas_dates_present": (
        calendar["date"].isin(christmas_dates).sum() == 4
    )
}

print("Holiday Merge QA")
print("=" * 50)

for check, result in holiday_merge_checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

In [ ]:
# ============================================================
# Step 3.5.6.1 — Create Sales Coverage Flags
# ============================================================

# Extract unique dates from the sales dataset.
# We need only one row per date for the calendar merge.

sales_dates = (
    train[["date"]]
    .drop_duplicates()
    .assign(has_sales_record=True)
)

# Merge sales coverage information into the calendar.
calendar = calendar.merge(
    sales_dates,
    on="date",
    how="left",
    validate="one_to_one"
)

# Dates without a matching sales record receive False.
calendar["has_sales_record"] = (
    calendar["has_sales_record"]
    .fillna(False)
    .astype(bool)
)

# Display the coverage result.
print("Sales coverage flag created.")

print(
    calendar["has_sales_record"].value_counts()
)

In [ ]:
# ============================================================
# Step 3.5.6.2 — Create Transaction Coverage Flags
# ============================================================

# Extract unique dates from the transaction dataset.

transaction_dates = (
    transactions[["date"]]
    .drop_duplicates()
    .assign(has_transaction_record=True)
)

# Merge transaction coverage into the calendar.

calendar = calendar.merge(
    transaction_dates,
    on="date",
    how="left",
    validate="one_to_one"
)

# Mark dates without transaction records as False.

calendar["has_transaction_record"] = (
    calendar["has_transaction_record"]
    .fillna(False)
    .astype(bool)
)

print("Transaction coverage flag created.")

print(
    calendar["has_transaction_record"].value_counts()
)

In [ ]:
# ============================================================
# Step 3.5.6.3 — Review Missing Coverage Dates
# ============================================================

coverage_gaps = calendar[
    (~calendar["has_sales_record"]) |
    (~calendar["has_transaction_record"])
][[
    "date",
    "has_sales_record",
    "has_transaction_record",
    "has_holiday_event",
    "has_holiday",
    "has_national_holiday"
]].copy()

print("Coverage gap dates:")

print(
    coverage_gaps.to_string(index=False)
)

In [ ]:
# ============================================================
# Step 3.5.6.4 — Coverage QA
# ============================================================

coverage_checks = {
    "calendar_rows_unchanged": len(calendar) == 1688,

    "calendar_dates_unique": (
        calendar["date"].nunique() == 1688
    ),

    "sales_dates_present": (
        calendar["has_sales_record"].sum() == 1684
    ),

    "transaction_dates_present": (
        calendar["has_transaction_record"].sum() == 1682
    ),

    "sales_gap_count": (
        (~calendar["has_sales_record"]).sum() == 4
    ),

    "transaction_gap_count": (
        (~calendar["has_transaction_record"]).sum() == 6
    )
}

print("Coverage QA")
print("=" * 50)

for check, result in coverage_checks.items():
    print(
        f"{check}: {'PASS' if result else 'FAIL'}"
    )

In [ ]:
# ============================================================
# Step 3.5.7.1 — Create Explicit Sales Closure Flag
# ============================================================

# Previously verified Christmas closure dates

christmas_closure_dates = pd.to_datetime([
    "2013-12-25",
    "2014-12-25",
    "2015-12-25",
    "2016-12-25"
])

# Identify dates satisfying the documented closure conditions

calendar["is_sales_closure"] = (
    calendar["date"].isin(christmas_closure_dates)
    & (~calendar["has_sales_record"])
    & calendar["has_national_holiday"]
)

print("Sales closure flag created.")

print(
    calendar["is_sales_closure"].value_counts()
)

In [ ]:
# ============================================================
# Step 3.5.7.2 — Review Sales Closure Dates
# ============================================================

closure_review = calendar[
    calendar["is_sales_closure"]
][[
    "date",
    "has_sales_record",
    "has_transaction_record",
    "has_holiday_event",
    "has_national_holiday",
    "is_sales_closure"
]]

print(closure_review.to_string(index=False))

In [ ]:
# ============================================================
# Step 3.5.7.3 — Review Unclassified Sales Gaps
# ============================================================

unclassified_sales_gaps = calendar[
    (~calendar["has_sales_record"])
    & (~calendar["is_sales_closure"])
][[
    "date",
    "has_holiday_event",
    "has_national_holiday"
]]

print(
    "Unclassified sales gap dates:",
    len(unclassified_sales_gaps)
)

print(
    unclassified_sales_gaps.to_string(index=False)
)

In [ ]:
# ============================================================
# Step 3.5.7.4 — Sales Closure QA
# ============================================================

closure_checks = {
    "closure_count_is_four": (
        calendar["is_sales_closure"].sum() == 4
    ),

    "all_closures_have_no_sales": (
        (~calendar.loc[
            calendar["is_sales_closure"],
            "has_sales_record"
        ]).all()
    ),

    "all_closures_are_national_holidays": (
        calendar.loc[
            calendar["is_sales_closure"],
            "has_national_holiday"
        ].all()
    ),

    "all_documented_christmas_dates_classified": (
        calendar.loc[
            calendar["date"].isin(christmas_closure_dates),
            "is_sales_closure"
        ].all()
    ),

    "no_unclassified_sales_gaps": (
        len(unclassified_sales_gaps) == 0
    ),

    "calendar_rows_unchanged": len(calendar) == 1688,

    "calendar_dates_unique": (
        calendar["date"].nunique() == 1688
    )
}

print("Sales Closure QA")
print("=" * 50)

for check, result in closure_checks.items():
    print(
        f"{check}: {'PASS' if result else 'FAIL'}"
    )

In [ ]:
# ============================================================
# Step 3.5.8.1 — Inspect Final Calendar Structure
# ============================================================

print("Calendar shape:", calendar.shape)

print("\nCalendar columns:")

for position, column in enumerate(calendar.columns, start=1):
    print(f"{position:02d}. {column}")

In [ ]:
# ============================================================
# Step 3.5.8.2 — Expected Calendar Schema
# ============================================================

expected_calendar_columns = [
    "date",
    "year",
    "quarter",
    "month",
    "month_name",
    "week",
    "day",
    "day_of_week",
    "day_name",
    "is_weekend",
    "days_from_start",
    "has_sales_record",
    "has_transaction_record",
    "has_holiday_event",
    "holiday_event_count",
    "has_national_holiday",
    "has_regional_holiday",
    "has_local_holiday",
    "has_holiday",
    "has_event",
    "has_additional",
    "has_bridge",
    "has_transfer",
    "has_work_day",
    "holiday_count",
    "event_count",
    "additional_count",
    "bridge_count",
    "transfer_count",
    "work_day_count",
    "is_sales_closure"
]

missing_columns = [
    column
    for column in expected_calendar_columns
    if column not in calendar.columns
]

unexpected_columns = [
    column
    for column in calendar.columns
    if column not in expected_calendar_columns
]

print("Expected columns:", len(expected_calendar_columns))
print("Actual columns:", len(calendar.columns))

print("\nMissing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

In [ ]:
# ============================================================
# Step 3.5.8.3 — Add Event Category Counts
# ============================================================

event_count_summary = (
    holiday_working
    .groupby("date")
    .agg(
        holiday_count=(
            "type",
            lambda x: (x == "Holiday").sum()
        ),
        event_count=(
            "type",
            lambda x: (x == "Event").sum()
        ),
        additional_count=(
            "type",
            lambda x: (x == "Additional").sum()
        ),
        bridge_count=(
            "type",
            lambda x: (x == "Bridge").sum()
        ),
        transfer_count=(
            "type",
            lambda x: (x == "Transfer").sum()
        ),
        work_day_count=(
            "type",
            lambda x: (x == "Work Day").sum()
        )
    )
    .reset_index()
)

print("Event category count summary:")
print(event_count_summary.shape)

print(event_count_summary.head())

In [ ]:
# ============================================================
# Step 3.5.8.4A — Merge Event Category Counts
# ============================================================

calendar = calendar.merge(
    event_count_summary,
    on="date",
    how="left",
    validate="one_to_one"
)

print("Event category counts merged.")

print("Rows:", len(calendar))
print("Columns:", len(calendar.columns))

In [ ]:
# ============================================================
# Step 3.5.8.4B — Fill Non-Event Category Counts
# ============================================================

event_count_columns = [
    "holiday_count",
    "event_count",
    "additional_count",
    "bridge_count",
    "transfer_count",
    "work_day_count"
]

calendar[event_count_columns] = (
    calendar[event_count_columns]
    .fillna(0)
    .astype("int64")
)

print("Non-event category counts handled.")

print(
    calendar[event_count_columns]
    .isna()
    .sum()
)

In [ ]:
# ============================================================
# Step 3.5.8.5 — Event Category Reconciliation
# ============================================================

source_event_counts = (
    holiday_working["type"]
    .value_counts()
)

calendar_event_counts = {
    "Holiday": calendar["holiday_count"].sum(),
    "Event": calendar["event_count"].sum(),
    "Additional": calendar["additional_count"].sum(),
    "Bridge": calendar["bridge_count"].sum(),
    "Transfer": calendar["transfer_count"].sum(),
    "Work Day": calendar["work_day_count"].sum()
}

print("Source vs Calendar Event Counts")
print("=" * 60)

for category, calendar_count in calendar_event_counts.items():

    source_count = source_event_counts.get(category, 0)

    status = (
        "PASS"
        if source_count == calendar_count
        else "FAIL"
    )

    print(
        f"{category}: "
        f"Source={source_count}, "
        f"Calendar={calendar_count}, "
        f"{status}"
    )

In [ ]:
# ============================================================
# Step 3.5.8.6 — Final Calendar Schema Verification
# ============================================================

expected_calendar_columns = [
    "date",
    "year",
    "quarter",
    "month",
    "month_name",
    "week",
    "day",
    "day_of_week",
    "day_name",
    "is_weekend",
    "days_from_start",
    "has_sales_record",
    "has_transaction_record",
    "has_holiday_event",
    "holiday_event_count",
    "has_national_holiday",
    "has_regional_holiday",
    "has_local_holiday",
    "has_holiday",
    "has_event",
    "has_additional",
    "has_bridge",
    "has_transfer",
    "has_work_day",
    "holiday_count",
    "event_count",
    "additional_count",
    "bridge_count",
    "transfer_count",
    "work_day_count",
    "is_sales_closure"
]

missing_columns = [
    col for col in expected_calendar_columns
    if col not in calendar.columns
]

unexpected_columns = [
    col for col in calendar.columns
    if col not in expected_calendar_columns
]

print("Expected columns:", len(expected_calendar_columns))
print("Actual columns:", len(calendar.columns))
print("Missing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

print(
    "\nSchema status:",
    "PASS"
    if not missing_columns and not unexpected_columns
    else "FAIL"
)

### Step 3.5.9 — Final Analytical Calendar QA

In [ ]:
# ============================================================
# Step 3.5.9.1 — Calendar Data Type Review
# ============================================================

print(calendar.dtypes)

In [ ]:
# ============================================================
# Step 3.5.9.2 — Calendar Missing-Value Review
# ============================================================

calendar_missing = calendar.isna().sum()

print("Missing values by column:")
print(calendar_missing)

print(
    "\nTotal missing cells:",
    calendar.isna().sum().sum()
)

In [ ]:
# ============================================================
# Step 3.5.9.3 — Final Structural QA
# ============================================================

final_calendar_checks = {
    "expected_row_count": len(calendar) == 1688,

    "expected_column_count": len(calendar.columns) == 31,

    "unique_dates": calendar["date"].nunique() == 1688,

    "no_duplicate_dates": calendar["date"].duplicated().sum() == 0,

    "date_range_correct": (
        calendar["date"].min() == sales_start
        and calendar["date"].max() == sales_end
    ),

    "dates_sorted": calendar["date"].is_monotonic_increasing,

    "no_missing_values": calendar.isna().sum().sum() == 0,

    "sales_coverage_correct": (
        calendar["has_sales_record"].sum() == 1684
    ),

    "transaction_coverage_correct": (
        calendar["has_transaction_record"].sum() == 1682
    ),

    "documented_closures_correct": (
        calendar["is_sales_closure"].sum() == 4
    ),

    "event_records_reconciled": (
        calendar["holiday_event_count"].sum()
        == len(holiday_working)
    )
}

print("FINAL ANALYTICAL CALENDAR QA")
print("=" * 60)

for check, result in final_calendar_checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print("=" * 60)

print(
    "Overall:",
    "PASS"
    if all(final_calendar_checks.values())
    else "REVIEW REQUIRED"
)

In [ ]:
# ============================================================
# Step 3.5.9.4 — Final Calendar Preview
# ============================================================

print("First 5 rows:")
display(calendar.head())

print("\nLast 5 rows:")
display(calendar.tail())

print("\nCalendar shape:")
print(calendar.shape)

In [ ]:
# ============================================================
# Step 3.5.10.1 — Standardize Boolean Indicator Types
# ============================================================

boolean_columns = [
    "has_holiday",
    "has_event",
    "has_additional",
    "has_bridge",
    "has_transfer",
    "has_work_day",
    "has_national_holiday",
    "has_regional_holiday",
    "has_local_holiday",
    "has_holiday_event",
    "has_sales_record",
    "has_transaction_record",
    "is_sales_closure",
    "is_weekend"
]

calendar[boolean_columns] = (
    calendar[boolean_columns]
    .astype("bool")
)

print("Boolean data types standardized.")

In [ ]:
# ============================================================
# Step 3.5.10.2 — Verify Boolean Types
# ============================================================

print(
    calendar[boolean_columns].dtypes
)

In [ ]:
# ============================================================
# Step 3.5.10.3 — Final Data Quality Check
# ============================================================

print("Boolean columns using bool dtype:",
      calendar[boolean_columns].dtypes.eq("bool").all())

print("Total missing cells:",
      calendar.isna().sum().sum())

print("Calendar shape:",
      calendar.shape)

In [ ]:
# ============================================================
# Step 3.5.11 — Export Analytical Calendar
# ============================================================

calendar_output_path = (
    PROCESSED_DIR / "analytical_calendar.csv"
)

calendar.to_csv(
    calendar_output_path,
    index=False
)

print("Analytical calendar exported successfully.")
print("Path:", calendar_output_path)
print("Rows:", len(calendar))
print("Columns:", len(calendar.columns))

In [ ]:
# ============================================================
# Step 3.5.12 — Reload Exported Calendar
# ============================================================

calendar_reloaded = pd.read_csv(
    calendar_output_path
)

calendar_reloaded["date"] = pd.to_datetime(
    calendar_reloaded["date"]
)

print("Reloaded calendar shape:")
print(calendar_reloaded.shape)

print("\nDate range:")
print(
    calendar_reloaded["date"].min(),
    "to",
    calendar_reloaded["date"].max()
)

print("\nDuplicate dates:",
      calendar_reloaded["date"].duplicated().sum())

print("\nMissing cells:",
      calendar_reloaded.isna().sum().sum())

In [ ]:
# ============================================================
# Step 3.5.12 — Exported Calendar QA
# ============================================================

export_checks = {
    "rows_correct": len(calendar_reloaded) == 1688,
    "columns_correct": len(calendar_reloaded.columns) == 31,
    "dates_unique": (
        calendar_reloaded["date"].nunique() == 1688
    ),
    "no_duplicate_dates": (
        calendar_reloaded["date"].duplicated().sum() == 0
    ),
    "date_range_correct": (
        calendar_reloaded["date"].min() == sales_start
        and calendar_reloaded["date"].max() == sales_end
    ),
    "no_missing_values": (
        calendar_reloaded.isna().sum().sum() == 0
    ),
    "sales_coverage_correct": (
        calendar_reloaded["has_sales_record"].sum() == 1684
    ),
    "transaction_coverage_correct": (
        calendar_reloaded["has_transaction_record"].sum() == 1682
    ),
    "sales_closures_correct": (
        calendar_reloaded["is_sales_closure"].sum() == 4
    ),
    "event_records_reconciled": (
        calendar_reloaded["holiday_event_count"].sum() == 286
    )
}

print("EXPORTED ANALYTICAL CALENDAR QA")
print("=" * 60)

for check, result in export_checks.items():
    print(
        f"{check}: {'PASS' if result else 'FAIL'}"
    )

print("=" * 60)

print(
    "Overall:",
    "PASS"
    if all(export_checks.values())
    else "REVIEW REQUIRED"
)

### Step 3.6.1 — Daily Sales Grain & Coverage Validation

In [ ]:
# Step 3.6.1 — Expected Daily Sales Grain

expected_store_count = train["store_nbr"].nunique()
expected_family_count = train["family"].nunique()

expected_daily_rows = (
    expected_store_count * expected_family_count
)

print("Expected store count:", expected_store_count)
print("Expected product-family count:", expected_family_count)
print("Expected rows per complete sales date:", expected_daily_rows)

In [ ]:
# Measure actual sales coverage for each date

daily_coverage = (
    train
    .groupby("date")
    .agg(
        row_count=("id", "size"),
        store_count=("store_nbr", "nunique"),
        family_count=("family", "nunique")
    )
    .reset_index()
)

daily_coverage.head()

In [ ]:
# Compare actual daily coverage with the expected grain

daily_coverage["expected_rows"] = expected_daily_rows

daily_coverage["rows_complete"] = (
    daily_coverage["row_count"]
    == expected_daily_rows
)

daily_coverage["stores_complete"] = (
    daily_coverage["store_count"]
    == expected_store_count
)

daily_coverage["families_complete"] = (
    daily_coverage["family_count"]
    == expected_family_count
)

daily_coverage["coverage_complete"] = (
    daily_coverage["rows_complete"]
    & daily_coverage["stores_complete"]
    & daily_coverage["families_complete"]
)

print("Total sales dates:", len(daily_coverage))
print(
    "Complete sales dates:",
    daily_coverage["coverage_complete"].sum()
)
print(
    "Incomplete sales dates:",
    (~daily_coverage["coverage_complete"]).sum()
)

In [ ]:
# Inspect dates with incomplete sales coverage

incomplete_daily_coverage = daily_coverage[
    ~daily_coverage["coverage_complete"]
].copy()

incomplete_daily_coverage

### Step 3.6.2 — Validate the actual sales grain

In [ ]:
# Step 3.6.2 — Validate Date × Store × Product Family grain

duplicate_grain_rows = train.duplicated(
    subset=["date", "store_nbr", "family"]
).sum()

print("Duplicate Date × Store × Family rows:", duplicate_grain_rows)

In [ ]:
# Compare actual sales rows with the theoretical complete grain

expected_complete_rows = (
    len(daily_coverage)
    * expected_store_count
    * expected_family_count
)

actual_sales_rows = len(train)

print("Sales dates:", len(daily_coverage))
print("Expected complete sales rows:", expected_complete_rows)
print("Actual sales rows:", actual_sales_rows)
print("Row-count reconciliation:",
      expected_complete_rows == actual_sales_rows)

In [ ]:
# Final grain validation

grain_validation = {
    "expected_stores": expected_store_count,
    "expected_families": expected_family_count,
    "expected_rows_per_date": expected_daily_rows,
    "sales_dates": len(daily_coverage),
    "incomplete_dates": int(
        (~daily_coverage["coverage_complete"]).sum()
    ),
    "duplicate_grain_rows": int(duplicate_grain_rows),
    "expected_total_rows": expected_complete_rows,
    "actual_total_rows": actual_sales_rows,
}

grain_validation

In [ ]:
# Step 3.6.3 — Sales value integrity

sales_missing = train["sales"].isna().sum()
sales_negative = (train["sales"] < 0).sum()
sales_zero = (train["sales"] == 0).sum()

print("Missing sales values:", sales_missing)
print("Negative sales values:", sales_negative)
print("Zero sales values:", sales_zero)

In [ ]:
# Inspect the sales-value distribution

sales_summary = train["sales"].describe()

sales_summary

In [ ]:
# Inspect the highest individual sales observations

top_sales_rows = (
    train[
        [
            "date",
            "store_nbr",
            "family",
            "sales",
            "onpromotion"
        ]
    ]
    .sort_values("sales", ascending=False)
    .head(10)
)

top_sales_rows

In [ ]:
# Establish control total before any aggregation

total_sales_before_aggregation = train["sales"].sum()

print(
    "Total Sales before aggregation:",
    total_sales_before_aggregation
)

### Step 3.6.4 — Investigate the high Sales values

In [ ]:
# Inspect historical sales statistics for the Store × Family
# combinations containing the top sales observations

top_combinations = (
    top_sales_rows[["store_nbr", "family"]]
    .drop_duplicates()
)

combination_history = (
    train.merge(
        top_combinations,
        on=["store_nbr", "family"],
        how="inner"
    )
    .groupby(["store_nbr", "family"])["sales"]
    .agg(
        observations="count",
        mean_sales="mean",
        median_sales="median",
        max_sales="max"
    )
    .reset_index()
)

combination_history

In [ ]:
# Show the top observations alongside historical
# Store × Family statistics

top_sales_context = (
    top_sales_rows
    .merge(
        combination_history,
        on=["store_nbr", "family"],
        how="left"
    )
)

top_sales_context

In [ ]:
# Promotion context for the highest sales observations

top_sales_context[
    [
        "date",
        "store_nbr",
        "family",
        "sales",
        "onpromotion"
    ]
]

In [ ]:
# Step 3.6.5 — Final Sales Data QA Gate

sales_qa = {
    "sales_dates": len(daily_coverage),
    "expected_stores": expected_store_count,
    "expected_families": expected_family_count,
    "expected_rows_per_date": expected_daily_rows,
    "incomplete_dates": int(
        (~daily_coverage["coverage_complete"]).sum()
    ),
    "duplicate_grain_rows": int(duplicate_grain_rows),
    "expected_total_rows": expected_complete_rows,
    "actual_total_rows": actual_sales_rows,
    "missing_sales_values": int(sales_missing),
    "negative_sales_values": int(sales_negative),
    "zero_sales_values": int(sales_zero),
    "total_sales_before_aggregation": total_sales_before_aggregation,
}

sales_qa

In [ ]:
# Formal QA decision

sales_qa_status = {
    "daily_coverage": (
        "PASS"
        if sales_qa["incomplete_dates"] == 0
        else "FAIL"
    ),
    "grain_uniqueness": (
        "PASS"
        if sales_qa["duplicate_grain_rows"] == 0
        else "FAIL"
    ),
    "row_reconciliation": (
        "PASS"
        if sales_qa["expected_total_rows"]
        == sales_qa["actual_total_rows"]
        else "FAIL"
    ),
    "sales_missing_values": (
        "PASS"
        if sales_qa["missing_sales_values"] == 0
        else "FAIL"
    ),
    "negative_sales": (
        "PASS"
        if sales_qa["negative_sales_values"] == 0
        else "FAIL"
    ),
    "extreme_values": "REVIEW",
}

sales_qa_status

In [ ]:
# Overall pre-aggregation decision

critical_checks = [
    sales_qa_status["daily_coverage"],
    sales_qa_status["grain_uniqueness"],
    sales_qa_status["row_reconciliation"],
    sales_qa_status["sales_missing_values"],
    sales_qa_status["negative_sales"],
]

overall_sales_qa = (
    "PASS"
    if all(status == "PASS" for status in critical_checks)
    else "FAIL"
)

print("Overall Sales QA:", overall_sales_qa)
print("Extreme-value review:", sales_qa_status["extreme_values"])

## Step 3.7 — Controlled Sales Aggregation

In [ ]:
# Step 3.7.1 — Define the controlled aggregation design

aggregation_design = {
    "source_grain": "Date × Store × Product Family",
    "target_grain": "Month × Total Retail Business",
    "measure": "Sales",
    "aggregation_function": "SUM",
    "forecast_target": "Monthly Total Sales",
    "sales_imputation": False,
    "christmas_closure_imputation": False,
    "zero_sales_retained": True,
}

aggregation_design

### Step 3.7.2 — Determine the expected monthly period range

In [ ]:
# Expected monthly periods for the forecasting dataset

expected_months = pd.date_range(
    start=train["date"].min(),
    end=train["date"].max(),
    freq="MS"
)

print("Expected monthly periods:", len(expected_months))
print("First month:", expected_months.min())
print("Last month:", expected_months.max())

### Step 3.7.3 — Identify the partial final month

In [ ]:
# Check the number of observed sales dates in each month

monthly_date_coverage = (
    train.assign(
        month=train["date"].dt.to_period("M")
    )
    .groupby("month")
    .agg(
        sales_dates=("date", "nunique"),
        first_sales_date=("date", "min"),
        last_sales_date=("date", "max")
    )
    .reset_index()
)

monthly_date_coverage.tail(6)

### Step 3.7.4 — Determine complete vs partial months

In [ ]:
# Classify monthly periods by observed date coverage

monthly_date_coverage["is_partial_month"] = (
    monthly_date_coverage["last_sales_date"]
    < monthly_date_coverage["month"].dt.to_timestamp("M")
)

monthly_date_coverage.tail(6)

In [ ]:
# Step 3.7.4 — Monthly calendar coverage validation

calendar_monthly = (
    calendar.assign(
        month=calendar["date"].dt.to_period("M")
    )
    .groupby("month")
    .agg(
        calendar_days=("date", "size"),
        observed_sales_days=("has_sales_record", "sum"),
        sales_closure_days=("is_sales_closure", "sum"),
        first_calendar_date=("date", "min"),
        last_calendar_date=("date", "max")
    )
    .reset_index()
)

# Identify the actual calendar month-end
calendar_monthly["month_end"] = (
    calendar_monthly["month"].dt.to_timestamp("M")
)

# Days accounted for by observed sales or documented closure
calendar_monthly["accounted_sales_days"] = (
    calendar_monthly["observed_sales_days"]
    + calendar_monthly["sales_closure_days"]
)

# Check whether the available calendar covers the complete month
calendar_monthly["is_partial_month"] = (
    calendar_monthly["last_calendar_date"]
    < calendar_monthly["month_end"]
)

# Check whether all available calendar days are accounted for
calendar_monthly["sales_coverage_complete"] = (
    calendar_monthly["accounted_sales_days"]
    == calendar_monthly["calendar_days"]
)

calendar_monthly.tail(6)

In [ ]:
# Step 3.7.5 — Monthly coverage QA

print("Total monthly periods:", len(calendar_monthly))

print(
    "Partial months:",
    calendar_monthly["is_partial_month"].sum()
)

print(
    "Complete calendar-coverage months:",
    (
        calendar_monthly["sales_coverage_complete"]
        & ~calendar_monthly["is_partial_month"]
    ).sum()
)

print(
    "Full months with unexplained coverage gaps:",
    (
        ~calendar_monthly["sales_coverage_complete"]
        & ~calendar_monthly["is_partial_month"]
    ).sum()
)

print("\nDecember coverage:")
display(
    calendar_monthly[
        calendar_monthly["month"].dt.month == 12
    ][
        [
            "month",
            "calendar_days",
            "observed_sales_days",
            "sales_closure_days",
            "sales_coverage_complete"
        ]
    ]
)

print("\nLatest month:")
display(calendar_monthly.tail(1))

In [ ]:
# Step 3.7.6 — Monthly Sales Aggregation

monthly_sales = (
    train.assign(
        month=train["date"].dt.to_period("M")
    )
    .groupby("month")
    .agg(
        total_sales=("sales", "sum"),
        source_rows=("sales", "size"),
        sales_dates_observed=("date", "nunique"),
        stores_observed=("store_nbr", "nunique"),
        families_observed=("family", "nunique")
    )
    .reset_index()
)

print("Monthly sales dataset shape:", monthly_sales.shape)

display(monthly_sales.head())
display(monthly_sales.tail())

In [ ]:
# Step 3.7.7 — Combine monthly Sales and calendar coverage

monthly_sales = monthly_sales.merge(
    calendar_monthly[
        [
            "month",
            "calendar_days",
            "sales_closure_days",
            "is_partial_month",
            "sales_coverage_complete"
        ]
    ],
    on="month",
    how="left",
    validate="one_to_one"
)

# Identify the periods eligible for initial full-month forecasting
monthly_sales["is_forecast_training_period"] = (
    ~monthly_sales["is_partial_month"]
    & monthly_sales["sales_coverage_complete"]
)

display(monthly_sales.tail(6))

In [ ]:
# Step 3.7.8 — Monthly Sales Control-Total Reconciliation

source_total_sales = train["sales"].sum()
monthly_total_sales = monthly_sales["total_sales"].sum()

sales_difference = monthly_total_sales - source_total_sales

monthly_sales_reconciliation = {
    "source_total_sales": source_total_sales,
    "monthly_total_sales": monthly_total_sales,
    "difference": sales_difference,
    "reconciliation_pass": abs(sales_difference) < 0.01,
    "monthly_periods": len(monthly_sales),
    "complete_months": int(
        monthly_sales["is_forecast_training_period"].sum()
    ),
    "partial_months": int(
        monthly_sales["is_partial_month"].sum()
    )
}

monthly_sales_reconciliation

In [ ]:
# Step 3.7.9 — Create the initial complete-month forecasting dataset

monthly_sales_complete = (
    monthly_sales[
        monthly_sales["is_forecast_training_period"]
    ]
    .copy()
    .reset_index(drop=True)
)

print("All monthly periods:", len(monthly_sales))
print("Complete forecasting periods:", len(monthly_sales_complete))
print(
    "Partial periods excluded from initial training:",
    monthly_sales["is_partial_month"].sum()
)

print("\nComplete-month date range:")
print(monthly_sales_complete["month"].min())
print(monthly_sales_complete["month"].max())

display(monthly_sales_complete.tail())

In [ ]:
# Step 3.7.10 — Monthly Sales Dataset QA

expected_month_sequence = pd.period_range(
    start="2013-01",
    end="2017-08",
    freq="M"
)

monthly_qa = {
    "expected_month_count": len(expected_month_sequence),
    "actual_month_count": len(monthly_sales),
    "unique_months": monthly_sales["month"].nunique(),
    "duplicate_months": monthly_sales["month"].duplicated().sum(),
    "month_sequence_correct": (
        monthly_sales["month"].tolist()
        == expected_month_sequence.tolist()
    ),
    "missing_sales_values": monthly_sales["total_sales"].isna().sum(),
    "negative_monthly_sales": (
        monthly_sales["total_sales"] < 0
    ).sum(),
    "forecast_training_months": (
        monthly_sales["is_forecast_training_period"].sum()
    ),
    "partial_months": monthly_sales["is_partial_month"].sum(),
    "partial_month_correct": (
        monthly_sales.loc[
            monthly_sales["is_partial_month"], "month"
        ].tolist()
        == [pd.Period("2017-08", freq="M")]
    ),
    "all_complete_months_coverage_valid": (
        monthly_sales.loc[
            ~monthly_sales["is_partial_month"],
            "sales_coverage_complete"
        ].all()
    ),
    "sales_reconciliation_pass": (
        abs(
            monthly_sales["total_sales"].sum()
            - train["sales"].sum()
        ) < 0.01
    )
}

monthly_qa

In [ ]:
monthly_qa_status = {
    "monthly_period_count": (
        "PASS"
        if monthly_qa["actual_month_count"] == 56
        else "FAIL"
    ),
    "unique_months": (
        "PASS"
        if monthly_qa["duplicate_months"] == 0
        and monthly_qa["unique_months"] == 56
        else "FAIL"
    ),
    "continuous_month_sequence": (
        "PASS"
        if monthly_qa["month_sequence_correct"]
        else "FAIL"
    ),
    "sales_values": (
        "PASS"
        if monthly_qa["missing_sales_values"] == 0
        and monthly_qa["negative_monthly_sales"] == 0
        else "FAIL"
    ),
    "partial_month_classification": (
        "PASS"
        if monthly_qa["partial_month_correct"]
        else "FAIL"
    ),
    "complete_month_coverage": (
        "PASS"
        if monthly_qa["all_complete_months_coverage_valid"]
        else "FAIL"
    ),
    "sales_control_total": (
        "PASS"
        if monthly_qa["sales_reconciliation_pass"]
        else "FAIL"
    )
}

monthly_qa_status

In [ ]:
# Step 3.7.11 — Export controlled monthly datasets

from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

monthly_sales_export = monthly_sales.copy()
monthly_complete_export = monthly_sales_complete.copy()

# Store month as a consistent YYYY-MM string in CSV
monthly_sales_export["month"] = (
    monthly_sales_export["month"].astype(str)
)

monthly_complete_export["month"] = (
    monthly_complete_export["month"].astype(str)
)

monthly_sales_path = (
    PROCESSED_DIR / "monthly_sales.csv"
)

monthly_complete_path = (
    PROCESSED_DIR / "monthly_sales_complete.csv"
)

monthly_sales_export.to_csv(
    monthly_sales_path,
    index=False
)

monthly_complete_export.to_csv(
    monthly_complete_path,
    index=False
)

print("Exported:", monthly_sales_path)
print("Exported:", monthly_complete_path)

In [ ]:
# Step 3.7.12 — Exported File QA

monthly_sales_reloaded = pd.read_csv(
    monthly_sales_path
)

monthly_complete_reloaded = pd.read_csv(
    monthly_complete_path
)

export_qa = {
    "all_months_rows": len(monthly_sales_reloaded),
    "complete_months_rows": len(monthly_complete_reloaded),
    "all_months_unique": (
        monthly_sales_reloaded["month"].is_unique
    ),
    "complete_months_unique": (
        monthly_complete_reloaded["month"].is_unique
    ),
    "all_months_sales_total_matches": (
        abs(
            monthly_sales_reloaded["total_sales"].sum()
            - train["sales"].sum()
        ) < 0.01
    ),
    "complete_months_end": (
        monthly_complete_reloaded["month"].iloc[-1]
    ),
    "partial_august_retained": (
        "2017-08" in monthly_sales_reloaded["month"].values
    ),
    "partial_august_excluded_from_training": (
        "2017-08" not in monthly_complete_reloaded["month"].values
    )
}

export_qa